# CodeForge AI — Fine-tune Qwen2.5-Coder-1.5B
### Run this on Kaggle with T4 GPU enabled

**Before running:**
1. Kaggle → Settings → Enable GPU (T4 x2)
2. Add dataset: `codeforge-training-data` (your uploaded train_data.jsonl)
3. Run all cells top to bottom
4. Training takes ~3-4 hours
5. Download `codeforge-qwen-1.5b.Q4_K_M.gguf` from output

In [ ]:
# Cell 1: Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
print('GPU check done')

In [ ]:
# Cell 2: Install Unsloth + dependencies
# This takes 3-5 minutes
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
print('Installation complete')

In [ ]:
# Cell 3: Load model
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048  # Context window — 2048 is good for code
DTYPE = None           # Auto-detect (bfloat16 on newer GPUs)
LOAD_IN_4BIT = True    # 4-bit quantization — fits in T4 memory

print('Loading Qwen2.5-Coder-1.5B...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)
print('Model loaded!')
print(f'Model parameters: {model.num_parameters():,}')

In [ ]:
# Cell 4: Add LoRA adapters
# LoRA lets us fine-tune only a small part of the model
# r=16 is a good balance of quality vs speed

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                          # LoRA rank — higher = more capacity
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha = 32,                 # Scaling factor (usually 2x rank)
    lora_dropout = 0.05,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',  # Saves memory
    random_state = 42,
    use_rslora = True,               # Rank-stabilized LoRA — better quality
)

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Cell 5: Load your dataset
import json
import os

# Kaggle datasets are mounted at /kaggle/input/
# Try multiple possible paths
POSSIBLE_PATHS = [
    '/kaggle/input/codeforge-training-data/train_data.jsonl',
    '/kaggle/input/train_data.jsonl',
    'train_data.jsonl',
]

data_path = None
for p in POSSIBLE_PATHS:
    if os.path.exists(p):
        data_path = p
        break

if not data_path:
    # List available datasets to help debug
    print('Dataset not found. Available files:')
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            print(os.path.join(root, f))
    raise FileNotFoundError('Upload train_data.jsonl to Kaggle as a dataset')

raw_data = []
with open(data_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            raw_data.append(json.loads(line))

print(f'Loaded {len(raw_data)} training examples from {data_path}')

In [ ]:
# Cell 6: Format data for Qwen2.5 chat template
# Qwen uses ChatML format: <|im_start|>role\ncontent<|im_end|>

SYSTEM_PROMPT = """You are CodeForge, an expert AI coding assistant specialized in:
- Python, JavaScript, TypeScript, Dart, and web technologies
- Indian startup stack: Django, Flutter, React, Node.js, Supabase, Railway, Razorpay
- WhatsApp API, Meta APIs, and mobile-first development
Write clean, efficient, well-commented code. Explain your approach briefly before the code."""

def format_example(example):
    instruction = example.get('instruction', '').strip()
    input_text = example.get('input', '').strip()
    output = example.get('output', '').strip()

    if not instruction or not output:
        return None

    # Combine instruction + input if input exists
    user_message = instruction
    if input_text:
        user_message = f'{instruction}\n\nAdditional context:\n{input_text}'

    # Format as ChatML
    formatted = (
        f'<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{user_message}<|im_end|>\n'
        f'<|im_start|>assistant\n{output}<|im_end|>'
    )
    return {'text': formatted}

# Format all examples
formatted_data = []
skipped = 0
for item in raw_data:
    result = format_example(item)
    if result:
        formatted_data.append(result)
    else:
        skipped += 1

print(f'Formatted: {len(formatted_data)} examples')
print(f'Skipped (empty):  {skipped}')
print()
print('Sample formatted example:')
print(formatted_data[0]['text'][:400] + '...')

In [ ]:
# Cell 7: Create HuggingFace Dataset
from datasets import Dataset

dataset = Dataset.from_list(formatted_data)

# Filter out sequences that are too long (would waste memory)
def check_length(example):
    tokens = tokenizer(example['text'], return_tensors='pt')
    return tokens['input_ids'].shape[1] <= MAX_SEQ_LENGTH

print('Filtering by sequence length...')
dataset = dataset.filter(check_length)
print(f'Dataset after filtering: {len(dataset)} examples')

# Split 95% train, 5% eval
split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']
print(f'Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

In [ ]:
# Cell 8: Configure Training
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = True,   # Packs multiple short examples into one sequence — faster training

    args = TrainingArguments(
        # Batch & gradient
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # Effective batch = 2*4 = 8

        # Epochs & steps
        num_train_epochs = 2,              # 2 full passes through data

        # Learning rate schedule
        learning_rate = 2e-4,
        lr_scheduler_type = 'cosine',
        warmup_ratio = 0.05,

        # Precision
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),

        # Logging & saving
        logging_steps = 25,
        eval_steps = 100,
        save_steps = 200,
        save_total_limit = 2,
        evaluation_strategy = 'steps',
        load_best_model_at_end = True,

        # Output
        output_dir = '/kaggle/working/checkpoints',
        report_to = 'none',

        # Optimization
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        max_grad_norm = 1.0,
        seed = 42,
    ),
)

print('Trainer configured!')
print(f'Training on {len(train_dataset)} examples for 2 epochs')

In [ ]:
# Cell 9: START TRAINING
# ~3-4 hours on T4 GPU
# Watch the loss go down — should start ~2.0, end ~0.8-1.2

print('Starting training...')
print('Expected time: 3-4 hours on T4 GPU')
print('Watch for: train_loss going down, eval_loss following')
print()

trainer_stats = trainer.train()

print()
print('=== TRAINING COMPLETE ===')
print(f'Total time: {trainer_stats.metrics["train_runtime"] / 3600:.2f} hours')
print(f'Final train loss: {trainer_stats.metrics["train_loss"]:.4f}')
print(f'Samples/second: {trainer_stats.metrics["train_samples_per_second"]:.2f}')

In [ ]:
# Cell 10: Quick test before saving
FastLanguageModel.for_inference(model)  # Switch to inference mode

test_prompt = """<|im_start|>system
You are CodeForge, an expert coding assistant.<|im_end|>
<|im_start|>user
Write a Python function to calculate GST amount for a given price and GST rate<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer(test_prompt, return_tensors='pt').to('cuda')
outputs = model.generate(
    **inputs,
    max_new_tokens = 300,
    temperature = 0.7,
    do_sample = True,
    pad_token_id = tokenizer.eos_token_id,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
# Print only the assistant response
assistant_part = response.split('<|im_start|>assistant')[-1]
print('=== MODEL TEST OUTPUT ===')
print(assistant_part.strip())

In [ ]:
# Cell 11: Save merged model
print('Merging LoRA adapters into base model...')
model.save_pretrained_merged(
    '/kaggle/working/codeforge-qwen-1.5b-merged',
    tokenizer,
    save_method = 'merged_16bit',
)
print('Merged model saved!')

In [ ]:
# Cell 12: Export to GGUF (for CodeForge / llama.cpp)
# Q4_K_M = best quality/size tradeoff (~1GB file)

print('Exporting to GGUF format...')
print('This is the file you will load in CodeForge')

model.save_pretrained_gguf(
    '/kaggle/working/codeforge-qwen-1.5b',
    tokenizer,
    quantization_method = 'q4_k_m',  # 4-bit quantized — ~1GB, best quality/size
)

import os
gguf_path = '/kaggle/working/codeforge-qwen-1.5b/model-Q4_K_M.gguf'
if os.path.exists(gguf_path):
    size_gb = os.path.getsize(gguf_path) / (1024**3)
    print(f'GGUF file created: {gguf_path}')
    print(f'File size: {size_gb:.2f} GB')
    print()
    print('NEXT STEP:')
    print('  Download this file from Kaggle output panel')
    print('  Copy to CodeForge models folder')
    print('  Load it in CodeForge just like any other model')
else:
    # Find the actual GGUF file
    for root, dirs, files in os.walk('/kaggle/working/codeforge-qwen-1.5b'):
        for f in files:
            if f.endswith('.gguf'):
                full_path = os.path.join(root, f)
                size_gb = os.path.getsize(full_path) / (1024**3)
                print(f'GGUF: {full_path} ({size_gb:.2f} GB)')

In [ ]:
# Cell 13: Also save LoRA adapters separately (optional)
# Useful if you want to continue training later without re-downloading the full model

model.save_pretrained('/kaggle/working/codeforge-qwen-1.5b-lora')
tokenizer.save_pretrained('/kaggle/working/codeforge-qwen-1.5b-lora')

print('LoRA adapters saved to: /kaggle/working/codeforge-qwen-1.5b-lora')
print()
print('=== ALL DONE ===')
print('Files in /kaggle/working/:')
for root, dirs, files in os.walk('/kaggle/working'):
    for f in files:
        full = os.path.join(root, f)
        size = os.path.getsize(full) / (1024**2)
        print(f'  {full.replace("/kaggle/working/", "")} ({size:.1f} MB)')